# Polymarket Data Explorer
Quick access to both data sources:
- **Source parquet** (raw Becker data)
- **Trade DB** (materialized DuckDB)
- **Catalog** (NautilusTrader parquet)

In [1]:
import duckdb
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

DATA_DIR = str(PROJECT_ROOT / '../prediction-market-analysis/data/polymarket')
TRADE_DB = '/var/folders/8b/pyl2g8gx1x7d8cp07k5xh6wh0000gn/T/polymarket_etl.duckdb'
CATALOG_DIR = str(PROJECT_ROOT / 'data/catalog')

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir exists: {Path(DATA_DIR).exists()}")
print(f"Trade DB exists: {Path(TRADE_DB).exists()}")
print(f"Catalog exists: {Path(CATALOG_DIR).exists()}")

Project root: /Users/ys/Desktop/projects/polymarket-bot
Data dir exists: True
Trade DB exists: True
Catalog exists: True


## 1. Source Markets (raw parquet)

In [2]:
con = duckdb.connect()
markets = con.execute(f"""
    SELECT condition_id, question, volume, active, closed, created_at
    FROM read_parquet('{DATA_DIR}/markets/*.parquet', union_by_name=true)
    ORDER BY volume DESC
    LIMIT 20
""").df()
print(f"Top 20 markets by volume:")
markets

Top 20 markets by volume:


,condition_id,question,volume,active,closed,created_at
0,0xdd22472e552920b8438158ea7238bfadfa4f736aa4ce...,Will Donald Trump win the 2024 US Presidential...,1.531479e+09,True,True,2024-01-05 04:33:51.332000+11:00
1,0xc6485bb7ea46d7bb89beb9c91e7572ecfc72a6273789...,Will Kamala Harris win the 2024 US Presidentia...,1.037039e+09,True,True,2024-01-05 04:40:17.792000+11:00
2,0xc3d4155148681756bfe67bb41d8d0882a8a122e7d376...,Will Donald Trump be inaugurated?,4.004095e+08,True,True,2024-11-02 07:59:58.040376+11:00
3,0x763cc84a3cde36b9ce12a8afe6d3e1e3e46e50692e8e...,Will the Sacramento Kings win the 2025 NBA Fin...,3.780115e+08,True,True,2024-09-25 01:33:30.214289+10:00
4,0x9872fe47fbf6284e5399c0f41d6d2c8fb310d2f4d2d5...,Will Nicolae Ciucă win the 2024 Romanian Presi...,3.265077e+08,True,True,2024-11-08 10:53:29.207545+11:00
5,0x655e5ca101c466b6293aa15e06173b78b293221803d5...,Will Zelenskyy wear a suit before July?,2.422312e+08,True,True,2025-05-23 08:19:20.138837+10:00
6,0x55c551896c10a74861f2fd88b4f928694310114704cc...,Will any other Republican Politician win the 2...,2.416551e+08,True,True,2024-01-07 06:52:35.394000+11:00
7,0x17815081230e3b9c78b098162c33b1ffa68c4ec29c12...,Fed decreases interest rates by 50+ bps after ...,2.350652e+08,True,True,2025-09-18 02:22:35.837625+10:00
8,0x7c6c69d91b21cbbea08a13d0ad51c0e96a956045aaad...,Fed increases interest rates by 25+ bps after ...,2.164557e+08,True,True,2025-09-18 02:22:39.874792+10:00
9,0x265366ede72d73e137b2b9095a6cdc9be6149290caa2...,Kamala Harris wins the popular vote?,1.637798e+08,True,True,2024-01-10 05:29:49.185000+11:00


In [3]:
# Market counts and date range
con.execute(f"""
    SELECT
        count(*) as total_markets,
        min(created_at) as earliest,
        max(created_at) as latest,
        count(*) FILTER (WHERE closed) as resolved,
        count(*) FILTER (WHERE active) as active
    FROM read_parquet('{DATA_DIR}/markets/*.parquet', union_by_name=true)
""").df()

,total_markets,earliest,latest,resolved,active
0,408863,2020-10-03 02:10:01.467000+10:00,2026-02-04 09:25:08.795666+11:00,383707,408863


## 2. Trade DB (materialized DuckDB)

In [4]:
tcon = duckdb.connect(TRADE_DB, read_only=True)
print("Tables:", [r[0] for r in tcon.execute('SHOW TABLES').fetchall()])
tcon.execute('SELECT count(*) as total_trades FROM trades').df()

Tables: ['trades']


,total_trades
0,404540000


In [5]:
# Sample trades
tcon.execute('SELECT * FROM trades LIMIT 10').df()

,token_id,side,price,size,maker,taker,block_timestamp
0,1431004624986814751264380688748638264892060670...,BUY,0.99,100.0000,0x4c0340f93E3c590118fc6E35215D51ce7Dc80CC1,0x4bFb41d5B3570DeFd03C39a9A4D8dE6Bd8B8982E,2023-03-13T21:52:58Z
1,8332325801828057364346058956381042156254694697...,SELL,0.99,100.0000,0x3Cf3E8d5427aED066a7A5926980600f6C3Cf87B3,0x4c0340f93E3c590118fc6E35215D51ce7Dc80CC1,2023-03-13T21:53:16Z
2,8332325801828057364346058956381042156254694697...,BUY,0.99,100.0000,0x4c0340f93E3c590118fc6E35215D51ce7Dc80CC1,0x4bFb41d5B3570DeFd03C39a9A4D8dE6Bd8B8982E,2023-03-13T21:53:16Z
3,1801342001508943638861985813934535432488977085...,BUY,0.40,500.0000,0x47339E4348665455B7Bf2fCc83738f75fA3d8864,0x47339E4348665455B7Bf2fCc83738f75fA3d8864,2023-03-13T21:54:00Z
4,8274163698713348509836471590441712628775533417...,BUY,0.60,500.0000,0x47339E4348665455B7Bf2fCc83738f75fA3d8864,0x4bFb41d5B3570DeFd03C39a9A4D8dE6Bd8B8982E,2023-03-13T21:54:00Z
5,1431004624986814751264380688748638264892060670...,SELL,0.99,115.0000,0x3Cf3E8d5427aED066a7A5926980600f6C3Cf87B3,0x4c0340f93E3c590118fc6E35215D51ce7Dc80CC1,2023-03-13T21:54:58Z
6,1431004624986814751264380688748638264892060670...,BUY,0.99,115.0000,0x4c0340f93E3c590118fc6E35215D51ce7Dc80CC1,0x4bFb41d5B3570DeFd03C39a9A4D8dE6Bd8B8982E,2023-03-13T21:54:58Z
7,1006096076251823826241931045489217749734454086...,SELL,0.06,41.3100,0xc412fDBe9956C2e242e8B45fae131c3E8598592d,0xfe016F9A1E480968cE409B3B3f88DF8A46906097,2023-03-13T22:13:13Z
8,5950497898383374283070528183684144466469822405...,BUY,0.93,150.0000,0xAaF4ACAFBb7980ce93631D238Eb10A5C4cA67883,0xfe016F9A1E480968cE409B3B3f88DF8A46906097,2023-03-13T22:13:13Z
9,5950497898383374283070528183684144466469822405...,BUY,0.92,150.2675,0xAaF4ACAFBb7980ce93631D238Eb10A5C4cA67883,0xfe016F9A1E480968cE409B3B3f88DF8A46906097,2023-03-13T22:13:13Z


In [6]:
# Trade date range and token count
tcon.execute("""
    SELECT
        count(*) as total_trades,
        count(DISTINCT token_id) as unique_tokens,
        min(block_timestamp) as earliest_trade,
        max(block_timestamp) as latest_trade
    FROM trades
""").df()

,total_trades,unique_tokens,earliest_trade,latest_trade
0,404540000,591183,2023-03-05T17:06:59Z,2026-01-25T17:26:06Z


In [7]:
# Top tokens by trade count
tcon.execute("""
    SELECT token_id, count(*) as n_trades,
           min(block_timestamp) as first_trade,
           max(block_timestamp) as last_trade
    FROM trades
    GROUP BY token_id
    ORDER BY n_trades DESC
    LIMIT 10
""").df()

,token_id,n_trades,first_trade,last_trade
0,2174263314346390629056905015582624153306727273...,3450272,2024-01-05T02:36:24Z,2024-11-06T15:20:35Z
1,6923692362007769102708394687114864697201113146...,1752210,2024-01-05T06:10:04Z,2024-11-06T18:07:20Z
2,4571487063409090840381374745821462554237605254...,1737244,2024-11-01T23:19:33Z,2025-01-20T22:53:10Z
3,4833104333661288389093875950949315923475504897...,1659347,2024-01-05T02:36:24Z,2024-11-06T15:20:33Z
4,8758495535924524640495212808245189728777857124...,1060734,2024-01-05T06:31:48Z,2024-11-06T18:07:20Z
5,8190334421393820036128586251940665003757639804...,763956,2024-11-01T23:19:33Z,2025-01-20T22:53:10Z
6,2127100029184336124920906570609716702908306732...,424270,2024-01-10T02:01:10Z,2024-11-12T09:56:45Z
7,3394546925096396354178105163799967772767263521...,398480,2025-04-22T18:34:55Z,2025-11-05T05:45:05Z
8,6856495334599070327400479404766108364224261454...,316699,2024-09-10T02:13:02Z,2025-04-27T21:06:04Z
9,4389801918844310925454401164414109574832743394...,266312,2024-01-10T02:01:10Z,2024-11-12T09:25:08Z


## 3. Catalog (NautilusTrader parquet)

In [8]:
# Instruments in the catalog
cat_con = duckdb.connect()
instruments = cat_con.execute(f"""
    SELECT id, outcome, activation_ns, expiration_ns
    FROM read_parquet('{CATALOG_DIR}/data/binary_option/**/*.parquet', union_by_name=true)
""").df()
print(f"{len(instruments)} instruments in catalog")
instruments.head(10)

20 instruments in catalog


,id,outcome,activation_ns,expiration_ns
0,1038641317947562855037344681972788901310803003...,Yes,1747952360138836992,1751284800000000000
1,1186216556675734598524047616448971821905673501...,Yes,1758126155837625088,1769558400000000000
2,1641964935406729841273691983077783073002667746...,Yes,1758126159874792192,1769558400000000000
3,2127100029184336124920906570609716702908306732...,Yes,1704824989184999936,1730808000000000000
4,2174263314346390629056905015582624153306727273...,Yes,1704389631332000000,1730808000000000000
5,3437958178989552856028121823975928023727730537...,No,1747952360138836992,1751284800000000000
6,4125767081787631735599723359611319519419460820...,Yes,1727192010214288896,1750680000000000000
7,4213984992957404608863078579678081372543591485...,No,1758126159874792192,1769558400000000000
8,4389801918844310925454401164414109574832743394...,No,1704824989184999936,1730808000000000000
9,4571487063409090840381374745821462554237605254...,Yes,1730494798040376064,1737374400000000000


## 4. Quick token lookup
Paste a token_id to inspect its trades.

In [9]:
# Pick a token to inspect (change this)
TOKEN_ID = tcon.execute("SELECT token_id FROM trades LIMIT 1").fetchone()[0]
print(f"Inspecting token: {TOKEN_ID[:40]}...")

trades_df = tcon.execute("""
    SELECT side, price, size, block_timestamp
    FROM trades
    WHERE token_id = $1
    ORDER BY block_timestamp
""", [TOKEN_ID]).df()

print(f"{len(trades_df)} trades")
trades_df.describe()

Inspecting token: 1431004624986814751264380688748638264892...
387 trades


,price,size
count,387.000000,387.000000
mean,0.498298,126.894095
std,0.240587,451.138979
min,0.040000,0.120000
25%,0.350000,15.820000
50%,0.420000,26.000000
75%,0.620000,100.000000
max,0.990000,5000.000000
